## Column & row operations — select, withColumn, filter, dedup

The everyday verbs of shaping a DataFrame — plus the one principle that ties them together.

**Column operations — four verbs:**

- **`select(...)`** — pick, reorder, or expression-evaluate columns; returns a new DataFrame.
- **`withColumn("new", expr)`** — add or replace *one* column; the cheapest way to derive.
- **`withColumnRenamed("old", "new")`** — rename in place; handy right before a join when both sides share a name.
- **`drop("c1", "c2")`** — remove columns.

**The principle: every transformation returns a *new* DataFrame.** Spark DataFrames are immutable — `df.withColumn(...)` does not change `df`, it returns a new DataFrame you bind to a variable. `selectExpr` lets you cast and derive in one SQL-flavoured call.

**Row operations:**

```python
df.filter(col("amount") > 1000)               # filter / where are aliases
df.where("status = 'approved' AND is_flagged = false")
df.distinct()                                  # drop full-row duplicates
df.dropDuplicates(subset=["transaction_id"])   # dupes on listed cols
```

`dropDuplicates` keeps an **unspecified** surviving row. For deterministic "keep latest" dedup — the pattern the bank's silver layer uses for retried upstream sends — rank with a window and keep row 1:

```python
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

w = Window.partitionBy("transaction_id").orderBy(col("ingested_at").desc())
deduped = df.withColumn("rn", row_number().over(w)).filter("rn = 1").drop("rn")
```

**Why the exam loves the window trick:** it's deterministic. `dropDuplicates` is not — which duplicate survives depends on partition order.
